In [2]:
import numpy as np
import argparse
import sys

from aie.iron import Kernel, ObjectFifo, Program, Runtime, Worker
from aie.iron.placers import SequentialPlacer
from aie.iron.device import NPU1Col1, NPU2
from aie.iron.controlflow import range_

In [49]:
def build_elementwise_robo(device, size):
    value_type = np.half
    input_type = np.ndarray[(3 * size + 4, ), np.dtype[value_type]]
    output_type = np.ndarray[(3 * size, ), np.dtype[value_type]]

    kernel_fn = Kernel(
        "elementwise_inc",
        "elementwise_incr.o",
        [input_type, output_type]
    )

    # FIXME: non funziona, vedere come si fa ad impacchettare un int in testa ad una fila di brainfloat
    of_in = ObjectFifo(input_type, name="input_fifo")
    of_out = ObjectFifo(output_type, name="output_fifo")

    # FIXME: non è parallelizzato: si attende che venga scritto tutto

    def core_fn(of_in, of_out, kernel):
        o = of_out.acquire(1)
        i = of_in.acquire(1)
        kernel_fn(i, o)
        of_in.release(1)
        of_out.release(1)

    worker = Worker(core_fn, [of_in.cons(), of_out.prod(), kernel_fn])

    rt = Runtime()

    with rt.sequence(input_type, output_type) as (i, o):
        rt.start(worker)
        rt.fill(of_in.prod(), i)
        rt.drain(of_out.cons(), o, wait=True)

    program = Program(device, rt)

    return program.resolve_program(SequentialPlacer())
    

In [50]:
with open("ewi.mlir", 'w') as f:
    print(build_elementwise_robo(NPU2(), 1024), file=f)

In [31]:
dir(build_elementwise_robo(NPU2(), 1024).operation)

['_CAPICreate',
 '_CAPIPtr',
 '__class__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 'attributes',
 'clone',
 'context',
 'create',
 'detach_from_parent',
 'erase',
 'get_asm',
 'location',
 'move_after',
 'move_before',
 'name',
 'operands',
 'operation',
 'opview',
 'parent',
 'parse',
 'print',
 'regions',
 'result',
 'results',
 'successors',
 'verify',
 'walk',
 'write_bytecode']

In [34]:
print(build_elementwise_robo(NPU2(), 1024))

module {
  aie.device(npu2) {
    %tile_0_2 = aie.tile(0, 2)
    %shim_noc_tile_0_0 = aie.tile(0, 0)
    aie.objectfifo @output_fifo(%tile_0_2, {%shim_noc_tile_0_0}, 2 : i32) : !aie.objectfifo<memref<3072xf16>> 
    aie.objectfifo @input_fifo(%shim_noc_tile_0_0, {%tile_0_2}, 2 : i32) : !aie.objectfifo<memref<3073xf16>> 
    func.func private @elementwise_inc(memref<3073xf16>, memref<3072xf16>)
    %core_0_2 = aie.core(%tile_0_2) {
      %c0 = arith.constant 0 : index
      %c9223372036854775807 = arith.constant 9223372036854775807 : index
      %c1 = arith.constant 1 : index
      scf.for %arg0 = %c0 to %c9223372036854775807 step %c1 {
        %0 = aie.objectfifo.acquire @output_fifo(Produce, 1) : !aie.objectfifosubview<memref<3072xf16>>
        %1 = aie.objectfifo.subview.access %0[0] : !aie.objectfifosubview<memref<3072xf16>> -> memref<3072xf16>
        %2 = aie.objectfifo.acquire @input_fifo(Consume, 1) : !aie.objectfifosubview<memref<3073xf16>>
        %3 = aie.objectfifo.subview.a